In [27]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from pydantic import BaseModel
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver

In [28]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages : Annotated[list[BaseMessage], add_messages]

In [29]:
model = ChatGoogleGenerativeAI(model= 'models/gemini-3.1-flash-lite-preview')

In [30]:
def chat_node(state: ChatState):

    message = state['messages']

    # prompt = ""

    response = model.invoke(message)

    state['messages'] = response

    return { "messages": [response]}


In [34]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

#nodes
graph.add_node("chat_node", chat_node)

#edges
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

workflow = graph.compile(checkpointer= checkpointer)


In [ ]:
# workflow = graph.compile()

# initial_state = { "messages": ["Who won the world cup recently in cricket"]}

# final_state = workflow.invoke(initial_state)['messages'][-1].content

# final_state

In [35]:
thread_id = "1"

while True:

    user_message = input("Type here: ")

    print(user_message)

    if user_message.strip().lower() in ['exit', 'bye', 'quit']:
        break

    config = {"configurable": {"thread_id" : "thread_id"}}

    response = workflow.invoke(
        {
            "messages": [
                HumanMessage(content= user_message)
            ]
            
        },
        config=config
    )

    # print("AI:",response['messages'][-1].content)
    content = response['messages'][-1].content
    text = content[0]['text']

    print("AI:", text)





hi my name is john
AI: Hi John! It's nice to meet you. How are you doing today? Is there anything I can help you with?
what is my name
AI: Your name is John!
how do you n=know my name
AI: I know your name because you told me just a moment ago! You said, "hi my name is john."

I store that information in our current conversation so that I can keep track of who I'm talking to.
exit
